In [1]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate openai gradio


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#imports

import time
from io import StringIO
import torch
import numpy as np
import pandas as pd
import random
from openai import OpenAI
from sqlalchemy import create_engine
import gradio as gr
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

e:\PROYECTOS GITHUB\Synthetic_DataSet_Generator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()  # busca .env en el directorio de trabajo
api_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HUGGINGFACE_TOKEN")

In [3]:
# Model Constants
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [6]:
# Authentication

if not hf_token or not api_key:
 raise ValueError("Missing HF_TOKEN or OPENAI_API_KEY. Set them as environment variables.")

login(hf_token, add_to_git_credential=True)
openai = OpenAI(api_key=api_key)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
# Tokenizer Setup

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

e:\PROYECTOS GITHUB\Synthetic_DataSet_Generator\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tomas\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [8]:
# Model Quantization for Performance Optimization

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [9]:
# Load Model Efficiency, this can take long since its dowloading the model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

Loading weights:   0%|          | 1/291 [00:00<01:28,  3.29it/s]e:\PROYECTOS GITHUB\Synthetic_DataSet_Generator\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
e:\PROYECTOS GITHUB\Synthetic_DataSet_Generator\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [06:51<00:00,  1.42s/it]


In [10]:
def generate_ev_driver(num_records, address_type):
    # Adjusting the prompt based on checkbox selection
    address_prompts = {
        "international": f"Generate {num_records} rows of synthetic personal data with international addresses and phone numbers.",
        "us_only": f"Generate {num_records} rows of synthetic personal data with U.S.-only addresses and phone numbers.",
        "us_international": f"Generate {num_records} rows of synthetic personal data with a mix of U.S. and international addresses and phone numbers.",
        "americas": f"Generate {num_records} rows of synthetic personal data with a mix of U.S., Canada, Central America, and South America addresses and phone numbers.",
        "europe": f"Generate {num_records} rows of synthetic personal data with Europe-only addresses and phone numbers.",
    }

    address_prompt = address_prompts.get(address_type, "Generate synthetic personal data.")
    # Generate unique driver IDs
    driver_ids = random.sample(range(1, 1000001), num_records)

    user_prompt = f"""
    {address_prompt}
    Each row should include:
    - driverid (unique from the provided list: {driver_ids})
    - first_name (string)
    - last_name (string)
    - email (string)
    - phone_number (string)
    - address (string)
    - city (string)
    - state (string)
    - zip_code (string)
    - country (string)

    Ensure the CSV format is valid, with proper headers and comma separation.
    """

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates structured CSV data."},
            {"role": "user", "content": user_prompt}
        ]
    )

    # Call the new function to clean and extract the CSV data
    return clean_and_extract_csv(response)

In [11]:
def clean_and_extract_csv(response):
    # Clean up the response and remove the last occurrence of the code block formatting
    csv_data = response.choices[0].message.content.strip()
    csv_data = csv_data.rsplit("```", 1)[0].strip()

    # Define header and split the content to extract the data
    header = "driverid,first_name,last_name,email,phone_number,address,city,state,zip_code,country"
    _, *content = csv_data.split(header, 1)

    # Return the cleaned CSV data along with the header
    return header + content[0].split("\n\n")[0] if content else csv_data

In [12]:
def update_dataset(num_records, address_type):
    response = generate_ev_driver(num_records, address_type)

    # Convert response to DataFrame
    try:
        df = pd.read_csv(StringIO(response))
    except Exception as e:
        return pd.DataFrame(), f"Error parsing dataset: {str(e)}"

    return df, response

In [13]:
# Function to handle address type selection
def check_address_selection(selected_type):
    if not selected_type:
        # Return the error message and set button to yellow and disabled
        return (
            "<span style='color:red;'>⚠️ Address type is required. Please select one.</span>",
            gr.update(interactive=False, elem_classes="yellow_btn")
        )
    # Return success message and set button to blue and enabled
    return (
        "<span style='color:green;'>Ready to generate dataset.</span>",
        gr.update(interactive=True, elem_classes="blue_btn")
    )


In [14]:
# Gradio UI
with gr.Blocks() as app:
    gr.Markdown("## Dynamic CSV Dataset Viewer")

    num_records_slider = gr.Slider(minimum=5, maximum=50, step=5, value=20, label="Number of Records")

    with gr.Row(equal_height=True):
        address_type_radio = gr.Radio(
            ["us_only", "international", "us_international", "americas", "europe"],
            value="",
            label="Address and Phone Type",
            info="Select the type of addresses and phone numbers"
        )
        status_text = gr.Markdown(
            "<span style='color:red;'>⚠️ Please select an address type above to proceed.</span>",
            elem_id="status_text"
        )

    generate_btn = gr.Button("Generate Data", interactive=True, elem_id="generate_btn")

    response_text = gr.Textbox(value="", label="Generated Driver List CSV", interactive=False)
    dataframe_output = gr.Dataframe(value=pd.DataFrame(), label="Generated Driver List Dataset")

    # Update status text and button style dynamically
    address_type_radio.change(fn=check_address_selection, inputs=[address_type_radio], outputs=[status_text, generate_btn])

    generate_btn.click(update_dataset, inputs=[num_records_slider, address_type_radio], outputs=[dataframe_output, response_text])

    # Custom CSS for button colors
    app.css = """
    .blue_btn {
        background-color: green;
        color: white;
    }
    """

app.launch(share=False)  # Ensure sharing is enabled in Colab

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
